In [1]:
import pandas as pd
import sys
sys.path.insert(0, "../../utils/")
from sklearn.model_selection import train_test_split
from training_models.regression_models import RegressionModels

In [2]:
def split(df_data, seed):
    #Separa los datos
    train_data, val_data = train_test_split(df_data, test_size=0.2, random_state=seed)
    return train_data, val_data

In [3]:
def train(train_v, validation_v, iteration, repr_name, div, seed):
    #Separa datos de sus target de entrenamiento y validacion
    train_values = train_v.drop(columns="target").values
    train_response = train_v["target"].values

    validation_values = validation_v.drop(columns="target").values
    validation_response = validation_v["target"].values

    print(f"Training model Random Forest, iteration: {iteration}")
    #Se instancia el objeto
    clf_model = RegressionModels(X_train=train_values, X_val=validation_values, y_train=train_response, y_val=validation_response)
    #Se entrena el respectivo algoritmo con k-fold
    clf_model.instance_random_forest()
    clf_model.process_model(kfold=True, k=5)

    #Se guarda el modelo
    dump(clf_model.model, f"../../models/RandomForest_regression_{div}_{iteration}_{repr_name}_seed{seed}.joblib")

    return clf_model.performances

In [4]:
def metrics(perf, iteration, seed, sampling):
    #Se obtienen las metricas de entrenamiento y validacion en variables diferentes
    train_metrics = perf["training_metrics"]
    val_metrics = perf["validation_metrics"]

    row = {
        "iteration": iteration,
        "seed": seed,
        "sampling": sampling
    }
    for metric_name in train_metrics:
        row[f"Train_{metric_name}"] = round(train_metrics[metric_name], 4)
        row[f"Val_{metric_name}"] = round(val_metrics[metric_name], 4)
    
    return row

In [5]:
def main_train(df_data, repr_name, unique_seeds, use_grid=False):
    all_metrics = []
    for i, seed in enumerate(unique_seeds):
        df_train, df_val= split(df_data, seed)
        perf_base = train(df_train, df_val, i, repr_name, seed)
        all_metrics.append(metrics(perf_base, i, seed))

    df_metrics = pd.DataFrame(all_metrics)
    df_metrics.to_csv(f"../../models/metrics_{repr_name}_reg_RandomForest.csv", index=False)

In [6]:
repr_name="ProtT5"
df_data = pd.read_csv(f"../../data/numerical_rep_reg/{repr_name}.csv")
df_data.drop(["experimental_characteristics"], axis=1, inplace=True)

FileNotFoundError: [Errno 2] No such file or directory: '../../data/numerical_rep_reg/ProtT5.csv'

In [ ]:
folder = "../../data/numerical_rep/"
unique_seeds= [42]
#unique_seeds = np.random.choice(range(100), size=30, replace=False)
#unique_seeds = [94, 42, 98, 43, 90, 44, 99, 93, 66, 34, 72, 60, 6, 39, 26, 74, 17,8, 51, 96, 53, 13, 20, 33, 29, 65, 46, 82, 79, 89]

In [ ]:
print(f"Processing {repr_name}")
metrics_path = f"../../models/metrics_regression_{repr_name}.csv"
seeds_used = unique_seeds
main_train(df_data, repr_name, seeds_used, use_grid=True)
print(f"Finished processing {repr_name}")
print("=====================================")